# 03 - 推理与测试

## 任务
- **所有模型**：Baseline / SFT / DPO / GRPO 统一推理对比
- **方案1**：对比 Standard / Zero-shot COT / Few-shot COT 的效果
- **生成提交文件**：输出 CSV 提交结果

## 运行环境
- CPU: 可跑通流程，速度慢
- GPU: 批量推理，速度快（推荐）

In [1]:
import os, sys

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("=== 调试信息 ===")
print(f"当前工作目录 (cwd): {os.getcwd()}")
print(f"当前文件: {__file__}" if '__file__' in locals() else "当前文件: 无法获取 (Jupyter 环境)")

# 最直接的方式：判断当前目录在哪里
cwd = os.getcwd()
workspace_root = None

if os.path.exists('/mnt/workspace/workspace/utils'):
    workspace_root = '/mnt/workspace/workspace'
elif os.path.exists('/mnt/workspace/utils'):
    workspace_root = '/mnt/workspace'
elif os.path.exists('../utils'):
    workspace_root = os.path.dirname(cwd)
elif os.path.exists('./utils'):
    workspace_root = cwd
elif 'workspace/notebooks' in cwd:
    workspace_root = os.path.dirname(cwd)
elif 'workspace' in cwd:
    workspace_root = cwd
else:
    workspace_root = cwd

sys.path.insert(0, workspace_root)
os.chdir(workspace_root)

print(f"\n设置后的工作目录: {os.getcwd()}")
print(f"sys.path 前两项: {sys.path[:2]}")
print(f"utils 目录是否存在: {os.path.exists(os.path.join(workspace_root, 'utils'))}")
print(f"utils 目录内容: {os.listdir(workspace_root) if os.path.exists(workspace_root) else '根目录不存在'}")

=== 调试信息 ===
当前工作目录 (cwd): /mnt/workspace
当前文件: 无法获取 (Jupyter 环境)

设置后的工作目录: /mnt/workspace/workspace
sys.path 前两项: ['/mnt/workspace/workspace', '/usr/local/lib/python312.zip']
utils 目录是否存在: True
utils 目录内容: ['baseline', 'data', 'notebooks', 'scheme1_cot', 'scheme2_data_enhancement', 'scheme3_dpo', 'scheme4_grpo', 'utils']


In [2]:
import subprocess, sys
for p in ['transformers', 'peft', 'tqdm']:
    try: __import__(p)
    except: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('依赖就绪')

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


依赖就绪


In [ ]:
from utils.pipeline_config import get_paths, get_device

paths = get_paths()
device = get_device()
BATCH_SIZE = 16 if device != 'cpu' else 1
MAX_NEW_TOKENS = 256  # DPO/GRPO 推理链较长，128 不够用

print(f"设备: {device}, 批大小: {BATCH_SIZE}")
print(f"模型: {paths['base_model']}")
print(f"数据: {paths['data_dir']}")
print(f"输出: {paths['output_dir']}")

import os
MODEL_PATHS = {
    'base': paths['base_model'],
    'scheme2_sft': "/root/primary_change/primary_math-main/workspace/outputs/scheme2_cot/final",
    'scheme3_dpo': "/root/primary_change/primary_math-main/workspace/outputs/scheme3_dpo/final",
    'scheme4_grpo': "/root/primary_change/primary_math-main/workspace/outputs/scheme4_grpo/final",
}

for name, path in MODEL_PATHS.items():
    if name == 'base':
        exists = '✓' if os.path.exists(path) else '✗'
    else:
        exists = '✓' if os.path.exists(path) else '✗'
    print(f"  {exists} {name}: {path}")

模型目录不存在: /mnt/models/qwen/Qwen2___5-0___5B-Instruct
尝试自动下载 Qwen2.5-0.5B-Instruct ...
模型下载目标目录: /mnt/workspace/models


2026-06-01 07:49:29,151 - modelscope - INFO - Target directory already exists, skipping creation.


模型已下载到: /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct
设备: cuda, 批大小: 16
模型: /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct
数据: /mnt/workspace/data
输出: /mnt/workspace/outputs
  ✓ base: /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct
  ✓ scheme2_sft: /mnt/workspace/outputs/scheme2_cot/final
  ✓ scheme3_dpo: /mnt/workspace/outputs/scheme3_dpo/final
  ✓ scheme4_grpo: /mnt/workspace/outputs/scheme4_grpo/final


## ===== 各方案模型准确率对比 =====

对所有方案（base、SFT、DPO、GRPO）逐一推理并计算准确率

In [4]:
from scheme1_cot.inference import load_model, inference_with_cot
from scheme1_cot.cot_prompts import create_messages_with_cot, get_cot_prompt
from utils.common import load_json, extract_number
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
import gc
import torch

print('推理模块导入成功')

推理模块导入成功


In [5]:
test_data = load_json(os.path.join(paths['data_dir'], 'test.json'))
train_data = load_json(os.path.join(paths['data_dir'], 'train.json'))

import random
random.seed(42)

if train_data and 'answer' in train_data[0]:
    eval_data = random.sample(train_data, min(600, len(train_data)))
    has_ground_truth = True
    print(f'使用 train.json 随机抽样600条做评估（有Ground Truth）')
else:
    eval_data = random.sample(test_data, min(400, len(test_data)))
    has_ground_truth = False
    print(f'使用 test.json 随机抽样400条做评估（无Ground Truth）')

print(f"评估样本: {len(eval_data)}条")
print(f"测试样本: {len(test_data)}条")

使用 train.json 随机抽样600条做评估（有Ground Truth）
评估样本: 600条
测试样本: 8000条


In [6]:
# 调试：检查数据结构
print("=== 数据结构调试 ===")
if eval_data:
    first_item = eval_data[0]
    print(f"第一条数据: {first_item}")
    print(f"第一条数据类型: {type(first_item)}")
    if isinstance(first_item, dict):
        print(f"字典key: {list(first_item.keys())}")
        for k, v in first_item.items():
            print(f"  {k}: {type(v)}, value: {v}")


=== 数据结构调试 ===
第一条数据: {'id': '10477', 'question': '每辆客车能坐36名乘客，25辆这样的车，可以为旅客提供多少个座位？', 'answer': '900', 'instruction': '这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。'}
第一条数据类型: <class 'dict'>
字典key: ['id', 'question', 'answer', 'instruction']
  id: <class 'str'>, value: 10477
  question: <class 'str'>, value: 每辆客车能坐36名乘客，25辆这样的车，可以为旅客提供多少个座位？
  answer: <class 'str'>, value: 900
  instruction: <class 'str'>, value: 这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。


In [7]:
all_model_results = {}
all_accuracies = {}

# 按推理顺序（从基础到高级）
model_order = ['base', 'scheme2_sft', 'scheme3_dpo', 'scheme4_grpo']

def load_cascaded_model(model_name, base_model_path, model_paths, device):
    """
    级联加载模型：正确处理 DPO/GRPO 需要前序 merge 的情况
    """
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path, use_fast=True, trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model_kwargs = {"trust_remote_code": True}
    if device != "cpu":
        model_kwargs["device_map"] = {"": device}
        model_kwargs["torch_dtype"] = torch.bfloat16
    else:
        model_kwargs["torch_dtype"] = torch.float32
    
    model = AutoModelForCausalLM.from_pretrained(base_model_path, **model_kwargs)
    if device == "cpu":
        model = model.to(device)
    
    if model_name == 'base':
        # Base 模型无需额外 PEFT
        pass
    elif model_name == 'scheme2_sft':
        # SFT 直接加在 base 上 → merge
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
    elif model_name == 'scheme3_dpo':
        # DPO 需要先加载 SFT → merge → 再加 DPO adapter → merge
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
        
        dpo_path = model_paths['scheme3_dpo']
        if os.path.exists(dpo_path) and os.path.exists(os.path.join(dpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, dpo_path)
            print(f"已加载 DPO PEFT: {dpo_path}")
            print(f"正在 merge DPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"DPO 权重已 merge")
    elif model_name == 'scheme4_grpo':
        # GRPO 需要 SFT → merge → DPO → merge → GRPO
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
        
        dpo_path = model_paths['scheme3_dpo']
        if os.path.exists(dpo_path) and os.path.exists(os.path.join(dpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, dpo_path)
            print(f"已加载 DPO PEFT: {dpo_path}")
            print(f"正在 merge DPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"DPO 权重已 merge")
        
        grpo_path = model_paths['scheme4_grpo']
        if os.path.exists(grpo_path) and os.path.exists(os.path.join(grpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, grpo_path)
            print(f"已加载 GRPO PEFT: {grpo_path}")
            print(f"正在 merge GRPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"GRPO 权重已 merge")
    
    # 全局设置 max_new_tokens，确保不受调用链影响
    model.generation_config.max_new_tokens = 256
    model.generation_config.max_length = None
    print(f"[DEBUG] generation_config: max_new_tokens={model.generation_config.max_new_tokens}, max_length={model.generation_config.max_length}")
    
    model.eval()
    return model, tokenizer

def run_inference(model, tokenizer, data, prompt_type='zero_shot',
                  batch_size=16, max_new_tokens=128):
    """自包含推理函数，直接控制 max_new_tokens，不依赖 inference.py"""
    from scheme1_cot.cot_prompts import create_messages_with_cot
    from utils.common import extract_number
    
    messages_list = [
        create_messages_with_cot(item["question"], prompt_type)
        for item in data
    ]
    
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    
    all_responses = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="批量推理"):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=512, padding_side="left").to(model.device)
        
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        for j, g in enumerate(generated):
            input_len = inputs["attention_mask"][j].sum().item()
            out = g[input_len:]
            all_responses.append(tokenizer.decode(out, skip_special_tokens=True).strip())
    
    results = []
    empty_count = 0
    for item, resp in zip(data, all_responses):
        answer = extract_number(resp)
        if not answer:
            answer = "0"
            empty_count += 1
        results.append((item["id"], answer))
    
    if empty_count > 0:
        print(f"  ⚠ {empty_count}/{len(data)} 条无有效数字，已填充 0")
    
    return results, all_responses

for model_name in model_order:
    model_path = MODEL_PATHS[model_name]
    
    # 检查模型是否存在
    if model_name == 'base':
        if not os.path.exists(MODEL_PATHS['base']):
            print(f"\n[!] {model_name} 模型不存在，跳过...")
            continue
    else:
        if not os.path.exists(model_path) or not os.path.exists(os.path.join(model_path, 'adapter_config.json')):
            print(f"\n[!] {model_name} PEFT 不存在，跳过...")
            continue
    
    print('\n' + '='*60)
    print(f"正在推理: {model_name}")
    print('='*60)
    
    # 加载模型（使用级联加载）
    print(f'加载模型: {paths["base_model"]}')
    model, tokenizer = load_cascaded_model(model_name, paths['base_model'], MODEL_PATHS, device)
    
    # 每个模型使用与训练一致的 prompt
    prompt_map = {
        'base': 'zero_shot',
        'scheme2_sft': 'sft',
        'scheme3_dpo': 'dpo',
        'scheme4_grpo': 'grpo',
    }
    infer_prompt = prompt_map.get(model_name, 'zero_shot')
    print(f'使用 prompt: {infer_prompt}')
    preds, raw_responses = run_inference(model, tokenizer, eval_data,
                                          prompt_type=infer_prompt,
                                          batch_size=BATCH_SIZE,
                                          max_new_tokens=MAX_NEW_TOKENS)
    all_model_results[model_name] = preds
    
    # 计算准确率
    if has_ground_truth:
        correct = sum(1 for (qid, pred), item in zip(preds, eval_data)
                     if pred.strip() == str(item['answer']).strip())
        acc = correct / len(preds) if preds else 0
        all_accuracies[model_name] = acc
        print(f"✓ 准确率: {acc:.2%} ({correct}/{len(preds)})")
    else:
        print(f"✓ 推理完成 (无 Ground Truth 无法计算准确率)")
    
    # 显示几个示例（含原始输出）
    print(f"  示例输出（前3条）:")
    for i in range(min(3, len(preds))):
        qid, pred = preds[i]
        raw = raw_responses[i][:200] if i < len(raw_responses) else ''
        print(f"    ID {qid}: 提取={pred} | 原始={raw}")
    
    # 清理显存
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


正在推理: base
加载模型: /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:08<00:00, 32.34it/s]


[DEBUG] generation_config: max_new_tokens=256, max_length=None
使用 prompt: zero_shot


批量推理: 100%|██████████| 38/38 [03:34<00:00,  5.65s/it]


✓ 准确率: 31.50% (189/600)
  示例输出（前3条）:
    ID 10477: 提取=25 | 原始=6名乘客，25辆这样的车，可以为旅客提供多少个座位？
assistant
步骤：首先计算总共有多少名乘客。
\[ 36 \, \text{人/辆} \times 25 \, \text{辆} = 900 \, \text{人} \]

答案：25辆这样的车，可以为旅客提供900个座位。
    ID 1824: 提取=18 | 原始=小明和他的8个好朋友每人都有两张画片，他们一共有多少张画片．
assistant
步骤：首先计算小明和他的8个朋友一共有的画片数量。因为每个小朋友都有2张画片，所以总共的画片数是 \(2 \times 9 = 18\) 张。

答案：他们一共有18张画片。
    ID 409: 提取=100 | 原始=个同学，第四辆车要坐多少个同学？
assistant
步骤：首先计算前三辆车总共乘坐了多少名同学。
- 第一、二、三辆车共乘坐的总人数为 \(82 \times 3 = 246\) 名同学。

然后，从总数中减去这3辆车已经乘坐的同学数，得到第四辆车需要乘坐的学生数。
- 四车一共是4辆车，所以需要乘坐的学生数为 \(346 - 246 = 100\) 名同学。

因此，第四辆车需要坐100个同学

正在推理: scheme2_sft
加载模型: /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2028.86it/s]


... (7 outputs left out)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2103.63it/s]


已加载 SFT PEFT: /mnt/workspace/outputs/scheme2_cot/final
正在 merge SFT 权重...
SFT 权重已 merge
已加载 DPO PEFT: /mnt/workspace/outputs/scheme3_dpo/final
正在 merge DPO 权重...
DPO 权重已 merge
已加载 GRPO PEFT: /mnt/workspace/outputs/scheme4_grpo/final
正在 merge GRPO 权重...
GRPO 权重已 merge
[DEBUG] generation_config: max_new_tokens=256, max_length=None
使用 prompt: grpo


批量推理: 100%|██████████| 38/38 [03:39<00:00,  5.77s/it]


✓ 准确率: 31.67% (190/600)
  示例输出（前3条）:
    ID 10477: 提取=900 | 原始=6名乘客，25辆这样的车，可以为旅客提供多少个座位？
assistant
1. 先计算25辆客车一共能坐多少人，用乘法计算
2. 每辆客车坐36名乘客，25辆客车的总人数是36乘以25
3. 分解计算：先算30乘以25等于750，再算6乘以25等于150，最后将两部分相加得到750加150等于900
4. 因此25辆客车总共可提供900个座位
答案：900
    ID 1824: 提取=18 | 原始=小明和他的8个好朋友每人都有两张画片，他们一共有多少张画片．
assistant
1. 先计算小明有多少张画片，他有2张画片
2. 再计算8个好朋友的总人数，每个朋友有2张画片，用乘法计算
3. 2乘以8等于16，所以共有16张画片
4. 将小明和好朋友的画片数量相加得到总数
5. 2加16等于18
答案：18
    ID 409: 提取=82 | 原始=个同学，第四辆车要坐多少个同学？
assistant
1. 先计算前3辆车一共坐了多少个同学，用乘法计算
2. 前3辆车共有82个同学，每车坐82个
3. 82乘以3等于246，所以前三辆车总共坐了246个同学
4. 总共有330个同学需要乘坐，减去前三辆车已坐的同学数
5. 330减246等于84，剩余的84个同学由第四辆车负责运送
6. 因此第四辆车需要坐84个同学
7. 检查题目描述是否正确


In [8]:
print('\n' + '='*60)
print('各方案准确率对比')
print('='*60)

if all_accuracies:
    # 按准确率排序显示
    sorted_models = sorted(all_accuracies.items(), key=lambda x: x[1], reverse=True)
    
    # 找最佳模型
    best_name, best_acc = sorted_models[0]
    
    print(f"\n排名:")
    for i, (name, acc) in enumerate(sorted_models):
        marker = '🏆' if i == 0 else '   '
        print(f"  {marker} {i+1:2d}. {name:12s}: {acc:.2%}")
    
    # 计算相对提升
    if 'base' in all_accuracies:
        base_acc = all_accuracies['base']
        print(f"\n相对基础模型提升:")
        for name in model_order:
            if name in all_accuracies and name != 'base':
                diff = all_accuracies[name] - base_acc
                sign = '+' if diff > 0 else ''
                print(f"  {name:12s}: {sign}{diff:.2%}")

    best_prompt = 'zero_shot'
    
else:
    # 没有 ground truth 时默认用最佳模型
    best_name = None
    for name in ['scheme4_grpo', 'scheme3_dpo', 'scheme2_sft', 'base']:
        if name in all_model_results:
            best_name = name
            break
    best_prompt = 'zero_shot'
    
    print(f'无法计算准确率（无 train.json）')
    print(f'默认使用模型: {best_name}')

print(f"\n选定最佳模型: {best_name}")


各方案准确率对比

排名:
  🏆  1. scheme2_sft : 47.67%
       2. scheme4_grpo: 31.67%
       3. base        : 31.50%
       4. scheme3_dpo : 29.50%

相对基础模型提升:
  scheme2_sft : +16.17%
  scheme3_dpo : -2.00%
  scheme4_grpo: +0.17%

选定最佳模型: scheme2_sft


## ===== 方案1：COT Prompt对比测试 =====

对比三种推理方式的效果差异（可选）

In [9]:
from scheme1_cot.cot_prompts import get_cot_prompt

# 只在有最佳模型时才做 Prompt 对比
if best_name in all_model_results:
    print(f'加载最佳模型: {best_name}')
    model, tokenizer = load_cascaded_model(best_name, paths['base_model'], MODEL_PATHS, device)
    
    if has_ground_truth:
        offline_results = {}
        for pt in ['standard', 'zero_shot', 'few_shot']:
            print(f'测试 {pt}...')
            preds, _ = run_inference(model, tokenizer, eval_data,
                                       prompt_type=pt,
                                       batch_size=BATCH_SIZE,
                                       max_new_tokens=MAX_NEW_TOKENS)
            correct = sum(1 for (qid, pred), item in zip(preds, eval_data)
                         if pred.strip() == str(item['answer']).strip())
            acc = correct / len(preds) if preds else 0
            offline_results[pt] = acc
            print(f'  {pt:15s}: {acc:.2%} ({correct}/{len(preds)})')
        
        best_prompt = max(offline_results, key=offline_results.get)
        print(f'✓ 最佳Prompt（离线验证）: {best_prompt} ({offline_results[best_prompt]:.2%})')
    else:
        best_prompt = 'zero_shot'
else:
    best_prompt = 'zero_shot'
    print('跳过 Prompt 对比')

加载最佳模型: scheme2_sft


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1967.05it/s]


已加载 SFT PEFT: /mnt/workspace/outputs/scheme2_cot/final
正在 merge SFT 权重...
SFT 权重已 merge
[DEBUG] generation_config: max_new_tokens=256, max_length=None
测试 standard...


批量推理: 100%|██████████| 38/38 [03:36<00:00,  5.69s/it]


  standard       : 50.00% (300/600)
测试 zero_shot...


批量推理: 100%|██████████| 38/38 [03:35<00:00,  5.68s/it]


  zero_shot      : 47.00% (282/600)
测试 few_shot...


批量推理: 100%|██████████| 38/38 [03:39<00:00,  5.77s/it]

  few_shot       : 43.83% (263/600)
✓ 最佳Prompt（离线验证）: standard (50.00%)


## ===== 生成提交文件 =====


In [10]:
# 重新加载最佳模型（确保模型对象有效）
print(f'加载模型: {best_name}')
model, tokenizer = load_cascaded_model(best_name, paths['base_model'], MODEL_PATHS, device)

print(f"使用模型: {best_name}")
print(f"使用Prompt: {best_prompt}")
print(f"测试数据: {len(test_data)}条")

final_results, _ = run_inference(model, tokenizer, test_data,
                                   prompt_type=best_prompt,
                                   batch_size=BATCH_SIZE,
                                   max_new_tokens=MAX_NEW_TOKENS)
print(f"推理完成: {len(final_results)}条")

加载模型: scheme2_sft


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2032.62it/s]


已加载 SFT PEFT: /mnt/workspace/outputs/scheme2_cot/final
正在 merge SFT 权重...
SFT 权重已 merge
[DEBUG] generation_config: max_new_tokens=256, max_length=None
使用模型: scheme2_sft
使用Prompt: standard
测试数据: 8000条


批量推理: 100%|██████████| 500/500 [44:54<00:00,  5.39s/it]

推理完成: 8000条


In [11]:
from utils.common import save_csv

SUBMIT_PATH = os.path.join(paths['output_dir'], 'submit.csv')
os.makedirs(paths['output_dir'], exist_ok=True)
save_csv(final_results, SUBMIT_PATH)

print(f"提交文件已保存: {SUBMIT_PATH}")

print('\n结果预览（前10条）:')
for i, (qid, ans) in enumerate(final_results[:10]):
    print(f"  {qid}: {ans}")

提交文件已保存: /mnt/workspace/outputs/submit.csv

结果预览（前10条）:
  0: 3960
  1: -18
  2: 32
  3: 101
  4: 6
  5: 1290
  6: 450
  7: 4000
  8: 8.8
  9: 18


In [12]:
print('\n' + '='*60)
print('推理与测试完成')
print('='*60)
print(f"提交文件: {SUBMIT_PATH}")
print(f"样本总数: {len(final_results)}")
print(f"使用模型: {best_name}")
print(f"使用Prompt: {best_prompt}")


推理与测试完成
提交文件: /mnt/workspace/outputs/submit.csv
样本总数: 8000
使用模型: scheme2_sft
使用Prompt: standard
